In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sksurv.metrics import concordance_index_ipcw, concordance_index_censored
from sksurv.util import Surv
from lifelines import KaplanMeierFitter
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

class TimeDiscretizer:
    def __init__(self, num_bins=50):
        self.num_bins = num_bins
        self.cuts = None
        
    def fit(self, times, events):
        uncensored = times[events == 1]
        try:
            _, self.cuts = pd.qcut(uncensored, q=self.num_bins, retbins=True, duplicates='drop')
        except:
            self.cuts = np.linspace(times.min(), times.max(), self.num_bins + 1)
        self.cuts[0] = 0
        self.cuts[-1] = max(times.max(), self.cuts[-1]) + 1e-5
        self.num_bins = len(self.cuts) - 1
        
    def transform(self, times):
        return np.clip(np.digitize(times, self.cuts) - 1, 0, self.num_bins - 1)


class SurvivalDataset(Dataset):
    def __init__(self, X, time_indices, events, durations):
        self.X = torch.FloatTensor(X)
        self.time_indices = torch.LongTensor(time_indices)
        self.events = torch.FloatTensor(events)
        self.durations = torch.FloatTensor(durations)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.time_indices[idx], self.events[idx], self.durations[idx]






df_ready_train = pd.read_csv("./data/data_processed/df_ready_train.csv")
df_ready_test = pd.read_csv("./data/data_processed/df_ready_test.csv")

TAU = 7

def is_binary(s):
    return set(s.dropna().unique()) <= {0, 1, '0', '1', True, False}

def encode(df):
    df = df.copy()
    for col in [c for c in df.select_dtypes(['object', 'bool']).columns if c != 'ID']:
        if is_binary(df[col]):
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype('int8')
        else:
            df[col] = df[col].astype('category').cat.codes.replace(-1, 0).astype('int16')
    return df

def softmax(x, axis=-1):
    x_max = np.max(x, axis=axis, keepdims=True)
    exp_x = np.exp(x - x_max)
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)

def relu(x):
    return np.maximum(x, 0.0)




class DeepHit:
    def __init__(self, input_dim, t_max=18, hidden_shared=88, hidden_cause=44, 
                 lr=1e-3, alpha=0.3, sigma=1.0, l2_reg=1e-4, seed=42):
        self.input_dim = input_dim
        self.t_max = t_max
        self.hs = hidden_shared
        self.hc = hidden_cause
        self.lr = lr
        self.alpha = alpha
        self.sigma = sigma
        self.l2_reg = l2_reg
        
        rng = np.random.default_rng(seed)
        
        self.W1 = rng.normal(0, np.sqrt(2.0/input_dim), size=(input_dim, hidden_shared))
        self.b1 = np.zeros(hidden_shared)
        
        cause_in = input_dim + hidden_shared
        self.W2 = rng.normal(0, np.sqrt(2.0/cause_in), size=(cause_in, hidden_cause))
        self.b2 = np.zeros(hidden_cause)
        
        self.W3 = rng.normal(0, np.sqrt(2.0/hidden_cause), size=(hidden_cause, t_max))
        self.b3 = np.zeros(t_max)
        
        self.beta1, self.beta2, self.eps = 0.9, 0.999, 1e-8
        self.t = 0
        
        self.m1, self.v1 = np.zeros_like(self.W1), np.zeros_like(self.W1)
        self.mb1, self.vb1 = np.zeros_like(self.b1), np.zeros_like(self.b1)
        self.m2, self.v2 = np.zeros_like(self.W2), np.zeros_like(self.W2)
        self.mb2, self.vb2 = np.zeros_like(self.b2), np.zeros_like(self.b2)
        self.m3, self.v3 = np.zeros_like(self.W3), np.zeros_like(self.W3)
        self.mb3, self.vb3 = np.zeros_like(self.b3), np.zeros_like(self.b3)
        
        self.rng = rng
        
    def forward(self, X):
        a1 = relu(X @ self.W1 + self.b1)
        z2 = np.column_stack([X, a1])
        a2 = relu(z2 @ self.W2 + self.b2)
        logits = a2 @ self.W3 + self.b3
        y = softmax(logits, axis=1)
        return y, (X, a1, z2, a2, logits)
    
    def backward(self, y, cache, times, events):
        X, a1, z2, a2, logits = cache
        N = len(times)
        eps = 1e-10
        
        times_clip = np.clip(times, 0, self.t_max - 1).astype(int)
        
        dy = np.zeros_like(y)
        loss_l1 = 0.0
        
        # L1
        for i in range(N):
            t = times_clip[i]
            e = events[i]
            
            if e == 1:
                prob = np.clip(y[i, t], eps, 1.0)
                loss_l1 -= np.log(prob)
                dy[i, t] -= 1.0 / prob
            else:
                cif = np.sum(y[i, :t+1])
                surv = np.clip(1 - cif, eps, 1.0)
                loss_l1 -= np.log(surv)
                dy[i, :t+1] += 1.0 / surv
        
        loss_l1 /= N
        
        # L2 
        loss_l2 = 0.0
        n_pairs = 0
        
        if self.alpha > 0:
            cif_all = np.cumsum(y, axis=1)
            event_idx = np.where(events == 1)[0]
            
            if len(event_idx) > 200:
                event_idx = self.rng.choice(event_idx, 200, replace=False)
            
            for i in event_idx:
                ti = times_clip[i]
                cif_i = cif_all[i, ti]
                
                survives = np.where(times > times[i])[0]
                
                if len(survives) > 0:
                    if len(survives) > 18:
                        survives = self.rng.choice(survives, 18, replace=False)
                    
                    cif_j = cif_all[survives, ti]
                    diff = cif_i - cif_j
                    exp_vals = np.exp(-diff / self.sigma)
                    
                    loss_l2 += np.sum(exp_vals)
                    n_pairs += len(survives)
                    
                    grad_i = -np.sum(exp_vals) / self.sigma
                    dy[i, :ti+1] += self.alpha * grad_i
                    
                    for j, exp_val in zip(survives, exp_vals):
                        dy[j, :ti+1] -= self.alpha * exp_val / self.sigma
        
        if n_pairs > 0:
            loss_l2 /= n_pairs
            dy /= n_pairs
        
        dy /= N
        
        # Backprop
        dy_sum = np.sum(y * dy, axis=1, keepdims=True)
        dlogits = y * (dy - dy_sum)
        
        dW3 = a2.T @ dlogits + 2 * self.l2_reg * self.W3
        db3 = np.sum(dlogits, axis=0)
        
        da2 = dlogits @ self.W3.T
        da2 = da2 * (a2 > 0)
        
        dW2 = z2.T @ da2 + 2 * self.l2_reg * self.W2
        db2 = np.sum(da2, axis=0)
        
        dz2 = da2 @ self.W2.T
        da1 = dz2[:, self.input_dim:]
        
        da1 = da1 * (a1 > 0)
        dW1 = X.T @ da1 + 2 * self.l2_reg * self.W1
        db1 = np.sum(da1, axis=0)
        
        reg_loss = self.l2_reg * (np.sum(self.W1**2) + np.sum(self.W2**2) + np.sum(self.W3**2))
        total_loss = loss_l1 + self.alpha * loss_l2 + reg_loss
        
        return dW1, db1, dW2, db2, dW3, db3, total_loss, loss_l1, loss_l2
    
    def adam_update(self, W, dW, m, v):
        m = self.beta1 * m + (1 - self.beta1) * dW
        v = self.beta2 * v + (1 - self.beta2) * (dW ** 2)
        m_hat = m / (1 - self.beta1 ** self.t)
        v_hat = v / (1 - self.beta2 ** self.t)
        W -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)
        return W, m, v
    
    def fit(self, X, times, events, epochs=150, patience=35, verbose=False):
        best_loss = np.inf
        no_improve = 0
        
        for epoch in range(epochs):
            self.t += 1
            
            y, cache = self.forward(X)
            grads = self.backward(y, cache, times, events)
            dW1, db1, dW2, db2, dW3, db3, loss, l1, l2 = grads
            
            self.W1, self.m1, self.v1 = self.adam_update(self.W1, dW1, self.m1, self.v1)
            self.b1, self.mb1, self.vb1 = self.adam_update(self.b1, db1, self.mb1, self.vb1)
            self.W2, self.m2, self.v2 = self.adam_update(self.W2, dW2, self.m2, self.v2)
            self.b2, self.mb2, self.vb2 = self.adam_update(self.b2, db2, self.mb2, self.vb2)
            self.W3, self.m3, self.v3 = self.adam_update(self.W3, dW3, self.m3, self.v3)
            self.b3, self.mb3, self.vb3 = self.adam_update(self.b3, db3, self.mb3, self.vb3)
            
            if loss < best_loss - 1e-5:
                best_loss = loss
                no_improve = 0
            else:
                no_improve += 1
                if no_improve >= patience:
                    if verbose:
                        print(f"Early stop @ epoch {epoch}")
                    break
            
            if verbose and epoch % 40 == 0:
                print(f"Epoch {epoch}: Loss={loss:.4f} (L1={l1:.4f}, L2={l2:.4f})")
        
        if verbose and no_improve < patience:
            print(f"Completed {epochs} epochs")
    
    def predict_risk(self, X, tau=TAU):
        y, _ = self.forward(X)
        cif = np.cumsum(y, axis=1)
        tau_idx = min(int(tau), self.t_max - 1)
        return cif[:, tau_idx]


# preprocessing 
print("Preprocessing")
train_long = encode(df_ready_train.copy())
test_long = encode(df_ready_test.copy())

train_long = train_long.replace([np.inf, -np.inf], np.nan).dropna(subset=['OS_YEARS', 'OS_STATUS'])

meta = ['ID', 'OS_YEARS', 'OS_STATUS']
features = [c for c in train_long.columns if c not in meta]

for col in features:
    if col not in test_long.columns:
        test_long[col] = 0

for df in (train_long, test_long):
    df[features] = df[features].apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan).fillna(0).astype(float)

features = [c for c in features if train_long[c].var() > 1e-4]

corr = train_long[features].corr().abs()
mask = np.triu(np.ones_like(corr, bool), k=1)
pairs = corr.where(mask).stack().loc[lambda s: s > .8]

drop = set()
mc = corr.mean()
for f1, f2, _ in pairs.reset_index().values:
    if f1 not in drop and f2 not in drop:
        drop.add(f1 if mc[f1] > mc[f2] else f2)
features = [c for c in features if c not in drop]

sel, cph = [], CoxPHFitter()
for col in features:
    tmp = train_long[[col, 'OS_YEARS', 'OS_STATUS']].rename(columns={'OS_YEARS': 'T', 'OS_STATUS': 'E'})
    try:
        cph.fit(tmp, 'T', 'E', show_progress=False)
        if cph.summary.loc[col, 'p'] < .05:
            sel.append(col)
    except:
        pass

print(f"{len(sel)} features\n")

X_raw_tr = train_long[sel].clip(train_long[sel].quantile(0.01), train_long[sel].quantile(0.99), axis=1)
X_raw_te = test_long[sel].clip(train_long[sel].quantile(0.01), train_long[sel].quantile(0.99), axis=1)

scaler = StandardScaler().fit(X_raw_tr)
X_train = scaler.transform(X_raw_tr)
X_test = scaler.transform(X_raw_te)

y_time = train_long['OS_YEARS'].astype(float).values
y_event = train_long['OS_STATUS'].astype(int).values

surv_all = Surv.from_arrays(event=y_event, time=y_time)


# grid search 
print("="*80)
print("GRID SEARCH INTELLIGENT")
print("="*80)

param_grid = {
    't_max': [16, 18, 20],           
    'hidden': [(80, 40), (88, 44), (96, 48)],  
    'lr': [8e-4, 1e-3, 1.2e-3],      
    'alpha': [0.25, 0.35, 0.45],     
    'l2_reg': [5e-5, 1e-4, 2e-4]     
}

results = []
total_configs = len(param_grid['t_max']) * len(param_grid['hidden']) * len(param_grid['lr']) * len(param_grid['alpha']) * len(param_grid['l2_reg'])

print(f"Total configurations: {total_configs}")
print(f"Test rapide: 150 epochs, patience=35\n")


config_num = 0
for t_max in param_grid['t_max']:
    for hidden in param_grid['hidden']:
        for lr in param_grid['lr']:
            for alpha in param_grid['alpha']:
                for l2_reg in param_grid['l2_reg']:
                    config_num += 1
                    
                    y_time_discrete = np.clip(np.floor(y_time).astype(int), 0, t_max - 1)
                    
                    model = DeepHit(
                        input_dim=X_train.shape[1],
                        t_max=t_max,
                        hidden_shared=hidden[0],
                        hidden_cause=hidden[1],
                        lr=lr,
                        alpha=alpha,
                        sigma=1.0,
                        l2_reg=l2_reg,
                        seed=42
                    )
                    
                    model.fit(X_train, y_time_discrete, y_event, 
                             epochs=150, patience=35, verbose=False)
                    
                    risk = model.predict_risk(X_train, tau=TAU)
                    score = concordance_index_ipcw(surv_all, surv_all, risk, tau=TAU)[0]
                    
                    results.append({
                        't_max': t_max,
                        'hidden': f"{hidden[0]},{hidden[1]}",
                        'lr': lr,
                        'alpha': alpha,
                        'l2_reg': l2_reg,
                        'score': score
                    })
                    
                    print(f"[{config_num}/{total_configs}] T={t_max} H={hidden} lr={lr:.0e} α={alpha} L2={l2_reg:.0e} → {score:.5f}")

# best result
df_results = pd.DataFrame(results).sort_values('score', ascending=False)

print("LES 10 MEILLEURES CONFIGURATIONS \n")
print(df_results.head(10).to_string(index=False))

best = df_results.iloc[0]
print("MEILLEURE CONFIGURATION")
for col in ['t_max', 'hidden', 'lr', 'alpha', 'l2_reg', 'score']:
    print(f"  {col:10s}: {best[col]}")

# final training
print("\n")
print("ENTRAÎNEMENT FINAL\n")


best_t_max = int(best['t_max'])
best_hidden = tuple(map(int, best['hidden'].split(',')))
best_lr = float(best['lr'])
best_alpha = float(best['alpha'])
best_l2 = float(best['l2_reg'])

N_RUNS = 3
final_scores = []
final_subs = []

for run in range(1, N_RUNS + 1):
    print(f"\n--- Run {run}/{N_RUNS} ---")
    
    y_time_discrete = np.clip(np.floor(y_time).astype(int), 0, best_t_max - 1)
    
    model = DeepHit(
        input_dim=X_train.shape[1],
        t_max=best_t_max,
        hidden_shared=best_hidden[0],
        hidden_cause=best_hidden[1],
        lr=best_lr,
        alpha=best_alpha,
        sigma=1.0,
        l2_reg=best_l2,
        seed=42 + run
    )
    
    model.fit(X_train, y_time_discrete, y_event, 
             epochs=220, patience=45, verbose=True)
    
    risk_train = model.predict_risk(X_train, tau=TAU)
    risk_test = model.predict_risk(X_test, tau=TAU)
    
    score = concordance_index_ipcw(surv_all, surv_all, risk_train, tau=TAU)[0]
    final_scores.append(score)
    print(f"C-index @{TAU}y: {score:.5f}")
    
    # Submisssion
    test_long['risk_row'] = risk_test
    sub = test_long.groupby('ID', as_index=False).agg(risk_score=('risk_row', 'mean'))
    sub['risk_score'] = (sub['risk_score'] - sub['risk_score'].min()) / \
                        (sub['risk_score'].max() - sub['risk_score'].min())
    final_subs.append((score, sub))

best_idx = np.argmax(final_scores)
best_final_score = final_scores[best_idx]
best_sub = final_subs[best_idx][1]

print("\n")
print("RÉSULTATS FINAUX")
print(f"Scores: {[f'{s:.5f}' for s in final_scores]}")
print(f"Moyenne: {np.mean(final_scores):.5f} ± {np.std(final_scores):.5f}")
print(f"Meilleur: {best_final_score:.5f}")

output = f"./submission/submission_deephit_optimized_{best_final_score:.5f}.csv"
best_sub.to_csv(output, index=False)

params = (f"DeepHit_GridSearch | T_MAX={best_t_max} | alpha={best_alpha} | "
          f"sigma=1.0 | hidden={best_hidden} | lr={best_lr:.0e} | "
          f"L2_reg={best_l2:.0e} | epochs=220 | patience=45 | features={len(sel)}")


Preprocessing
17 features

GRID SEARCH INTELLIGENT
Total configurations: 243
Test rapide: 150 epochs, patience=35

[1/243] T=16 H=(80, 40) lr=8e-04 α=0.25 L2=5e-05 → 0.64310
[2/243] T=16 H=(80, 40) lr=8e-04 α=0.25 L2=1e-04 → 0.59034
[3/243] T=16 H=(80, 40) lr=8e-04 α=0.25 L2=2e-04 → 0.54236
[4/243] T=16 H=(80, 40) lr=8e-04 α=0.35 L2=5e-05 → 0.64383
[5/243] T=16 H=(80, 40) lr=8e-04 α=0.35 L2=1e-04 → 0.59315
[6/243] T=16 H=(80, 40) lr=8e-04 α=0.35 L2=2e-04 → 0.54309
[7/243] T=16 H=(80, 40) lr=8e-04 α=0.45 L2=5e-05 → 0.64251
[8/243] T=16 H=(80, 40) lr=8e-04 α=0.45 L2=1e-04 → 0.59510
[9/243] T=16 H=(80, 40) lr=8e-04 α=0.45 L2=2e-04 → 0.54346
[10/243] T=16 H=(80, 40) lr=1e-03 α=0.25 L2=5e-05 → 0.66080
[11/243] T=16 H=(80, 40) lr=1e-03 α=0.25 L2=1e-04 → 0.60782
[12/243] T=16 H=(80, 40) lr=1e-03 α=0.25 L2=2e-04 → 0.54835
[13/243] T=16 H=(80, 40) lr=1e-03 α=0.35 L2=5e-05 → 0.65926
[14/243] T=16 H=(80, 40) lr=1e-03 α=0.35 L2=1e-04 → 0.61224
[15/243] T=16 H=(80, 40) lr=1e-03 α=0.35 L2=2e-04 → 0.